# 01 - Confounding and propensity scores

This notebook shows why a naive comparison can be biased in observational data and how propensity scores can reduce that bias under the usual adjustment assumptions.


## Causal question
State the causal question this notebook answers before reading numeric output.

## Causal setup
- Treatment: define the treatment variable and intervention of interest.
- Outcome: define the outcome variable being affected by treatment.
- Covariates: list observed confounders included in the design/diagnostics.
- Unit of analysis: specify the observational unit used in this notebook.

## Estimand
Specify the target estimand (ATE, ATT, CATE, etc.) and how it maps to model coefficients.

## Identification assumptions
Enumerate the identification assumptions required for a causal interpretation (for example ignorability, overlap, no interference).

## Uncertainty and limitations
Report interval estimates, sensitivity checks, and at least one limitation of the design.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Step execution
This cell performs the next computation. Read the result and tie it back to the causal setup before moving on.


In [ ]:
from causal_inference_lab.data_generators import make_confounded_binary_treatment
from causal_inference_lab.estimators import difference_in_means, ipw_ate, estimate_propensity_scores
from causal_inference_lab.diagnostics import balance_table, ipw_weights, overlap_summary
from causal_inference_lab.plotting import plot_propensity_overlap, plot_balance_table
from causal_inference_lab.matching import matching_balance_table, nearest_neighbour_matching, propensity_score_matching

dataset = make_confounded_binary_treatment(n=5_000, seed=42)
data = dataset.data
covariates = ["x1", "x2", "x3"]

data.head()


## Naive comparison

The naive difference in means compares treated and untreated units directly. In this data, treatment assignment depends on the same covariates that affect the outcome, so this estimate is biased.


In [ ]:
naive = difference_in_means(data)
print(f"Naive estimate: {naive.estimate:.3f}")
print(f"True ATE:        {dataset.true_ate:.3f}")
print(f"Naive error:    {abs(naive.estimate - dataset.true_ate):.3f}")


## Propensity score model

The propensity score is the probability of receiving treatment conditional on observed covariates.

Identification assumption: after conditioning on the observed covariates, treatment assignment is as-if random.


In [ ]:
propensity_scores = estimate_propensity_scores(data, covariates)
overlap_summary(propensity_scores)


## Step execution
This cell performs the next computation. Read the result and tie it back to the causal setup before moving on.


In [ ]:
fig = plot_propensity_overlap(data, propensity_scores)
plt.show()


## Balance before and after weighting

Standardized mean differences help check whether treated and control groups are comparable on observed covariates.


In [ ]:
before = balance_table(data, covariates)
weights = ipw_weights(data, covariates)
after = balance_table(data, covariates, weights=weights)

comparison = before.merge(after, on="covariate", suffixes=("_before", "_after"))
comparison


## Step execution
This cell performs the next computation. Read the result and tie it back to the causal setup before moving on.


In [ ]:
fig = plot_balance_table(before)
plt.show()

fig = plot_balance_table(after)
plt.show()


## IPW estimate

IPW reweights observations to create a pseudo-population where treatment is less associated with the observed covariates.


In [ ]:
ipw = ipw_ate(data, covariates)

print(f"IPW estimate: {ipw.estimate:.3f}")
print(f"True ATE:     {dataset.true_ate:.3f}")
print(f"IPW error:    {abs(ipw.estimate - dataset.true_ate):.3f}")


## Matching as an interpretable alternative

Matching can make the adjustment logic transparent: each treated unit is paired with a comparable control.


In [ ]:
ps_match = propensity_score_matching(data, covariates)
ps_match_att = ps_match.effect.estimate
nn_match = nearest_neighbour_matching(data, covariates)
nn_match_att = nn_match.effect.estimate
before_balance, after_ps_balance = matching_balance_table(
    data=data,
    covariates=covariates,
    matched_data=ps_match.matched_data,
)
before_mean_abs_smd = float(before_balance["abs_smd"].abs().mean())
after_mean_abs_smd = float(after_ps_balance["abs_smd"].abs().mean())

print(f"Propensity matching ATT: {ps_match_att:.3f}")
print(f"Nearest-neighbour ATT: {nn_match_att:.3f}")
print(f"Mean abs SMD before matching: {before_mean_abs_smd:.3f}")
print(f"Mean abs SMD after matching:  {after_mean_abs_smd:.3f}")
print(f"Dropped units from matching:    {ps_match.dropped_units}")


## Matching interpretation

Matching reduces measured imbalance, but residual imbalance or weak overlap means this strategy can still fail.


**Interpretation.** IPW improves the estimate because the treatment model uses the confounders. This does not prove causality by itself. It only supports the analysis if the adjustment set is credible and overlap is acceptable.
